# 06. RNN — 순서가 있는 데이터

노트북 03~05는 전부 **이미지**였다. 이미지는 **공간(space)** 데이터 — 픽셀이 상하좌우로 놓인다.
CNN이 그 공간 구조를 다뤘다.

이 노트북은 **순서(sequence)** 데이터를 다룬다. 순서가 의미를 가지는 데이터들이다:
- **글자**: "cat"과 "act"는 같은 글자지만 순서가 다르면 다른 단어
- **문장**: 단어의 순서가 의미를 만든다
- **시계열**: 주가, 날씨처럼 시간 순서가 있는 값

**RNN(순환 신경망, Recurrent Neural Network)** 은 순서를 **하나씩 차례로** 읽으면서,
**앞에서 본 것을 기억(state)** 해 다음 판단에 쓴다. 이 "기억"이 순서 데이터의 핵심이다.

이 노트북의 세 실험:
1. **문자 예측** — 'hihello'에서 다음 글자 맞히기
2. **감성 분석** — 짧은 문장이 긍정인지 부정인지
3. **MNIST를 시퀀스로** — 이미지를 "위에서 아래로 한 줄씩 읽는" 순서 데이터로 취급


## RNN은 어떻게 "기억"하는가

일반 신경망(DNN)은 입력을 **한 번에** 받는다. RNN은 **한 스텝씩** 받는다.

```
스텝1: 'h' 입력 → [RNN] → 기억1 ─┐
스텝2: 'i' 입력 → [RNN] → 기억2 ─┤  각 스텝에서 이전 기억을
스텝3: 'h' 입력 → [RNN] → 기억3 ─┘  함께 받아 다음 기억을 만든다
...
```

핵심은 **같은 RNN 셀이 매 스텝 반복(recurrent)** 된다는 것이다.
각 스텝에서 셀은 두 가지를 받는다 — **현재 입력** + **직전 스텝의 기억(hidden state)**.
그래서 "지금까지 뭘 봤는지"가 다음 판단에 반영된다.

Keras에서는 `SimpleRNN(units)` 로 쓴다. `units`는 기억(state)의 크기다.

- `return_sequences=False` (기본): **마지막 스텝의 기억 하나**만 출력 → 문장 전체를 하나로 요약할 때 (감성분석)
- `return_sequences=True`: **매 스텝의 출력을 전부** 내보냄 → 글자마다 다음 글자를 예측할 때 (문자 예측), 또는 RNN을 여러 층 쌓을 때


## 1부. 문자 예측 — 'hihello'

가장 작은 예제. 문자열 `'hihello'`에서 **각 글자 다음에 올 글자**를 예측하게 한다.

```
입력: h i h e l l    (앞 6글자)
정답: i h e l l o    (뒤 6글자, 한 칸 밀림)
```

즉 "h 다음엔 i", "i 다음엔 h", ... "l 다음엔 o"를 배우는 것이다.

### 1-1. 글자를 숫자로

모델은 글자를 못 받으니 숫자로 바꾼다. `'hihello'`의 고유 글자는 {h, i, e, l, o} 5개.
각 글자에 번호를 매기고(`ch2id`), 다시 원-핫으로 만든다.

> `sorted(set(txt))` : `set`은 순서가 매번 달라지므로 `sorted`로 고정한다.
> (원본 수업은 `set(txt)`만 써서 실행마다 번호가 바뀌었다 — 재현성을 위해 정렬한다.)


In [1]:
import numpy as np
from tensorflow.keras.utils import to_categorical

txt = 'hihello'
chars = sorted(set(txt))            # 정렬로 순서 고정: ['e','h','i','l','o']
ch2id = {ch: i for i, ch in enumerate(chars)}
id2ch = {i: ch for i, ch in enumerate(chars)}
print("글자→번호:", ch2id)

txt_id = [ch2id[ch] for ch in txt]
print("txt_id:", txt_id)

data = to_categorical(txt_id, num_classes=len(chars))
x_data = data[:-1][np.newaxis, :, :]   # 앞 6글자 (1, 6, 5)
y_data = data[1:][np.newaxis, :, :]    # 뒤 6글자 (1, 6, 5)
print("x:", x_data.shape, "y:", y_data.shape)


글자→번호: {'e': 0, 'h': 1, 'i': 2, 'l': 3, 'o': 4}
txt_id: [1, 2, 1, 0, 3, 3, 4]
x: (1, 6, 5) y: (1, 6, 5)


### 1-2. 문자 예측 RNN

```
Input(6, 5)  →  SimpleRNN(16, return_sequences=True)  →  Dense(5, softmax)
   6스텝×5차원      매 스텝 기억 출력                        각 스텝마다 다음 글자 확률
```

- `return_sequences=True` : 6글자 각각에 대해 다음 글자를 예측해야 하므로 **매 스텝 출력**이 필요.
- 데이터가 1개(문장 하나)뿐이라 이건 **암기**에 가깝다. RNN이 순서를 학습하는 원리를 보는 게 목적.


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dense

model = Sequential()
model.add(Input(shape=(6, 5)))
model.add(SimpleRNN(16, return_sequences=True))
model.add(Dense(5, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_data, y_data, epochs=200, verbose=0)   # 200에폭, 로그 생략
print("학습 완료")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 6, 16)          │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6, 5)           │            85 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 437 (1.71 KB)

 Trainable params: 437 (1.71 KB)

 Non-trainable params: 0 (0.00 B)

학습 완료


### 1-3. 예측 결과

학습된 모델에 다시 'hihello'의 앞 6글자를 넣고, 각 스텝의 예측 글자를 확인한다.
잘 배웠다면 `i h e l l o`가 나와야 한다 ('hihello'에서 첫 글자 뺀 것).


In [3]:
pred = model.predict(x_data, verbose=0)
result = [id2ch[i] for i in np.argmax(pred[0], axis=1)]
print("입력:", list(txt[:-1]))
print("예측:", result)
print("정답:", list(txt[1:]))


입력: ['h', 'i', 'h', 'e', 'l', 'l']
예측: ['i', 'h', 'e', 'l', 'l', 'o']
정답: ['i', 'h', 'e', 'l', 'l', 'o']


## 2부. 감성 분석 — 문장의 긍정/부정

이번엔 **문장 단위**. 4개의 짧은 문장이 긍정인지 부정인지 분류한다.

```
'i love you'  → 긍정(1)
'i hate you'  → 부정(0)
'i like you'  → 긍정(1)
'i curse you' → 부정(0)
```

문자 예측과 다른 점: **문장 전체를 읽고 마지막에 하나의 답**(긍/부정)을 낸다.
그래서 `return_sequences=False`(기본) — 마지막 스텝의 기억만 쓴다.

### 2-1. 단어를 숫자로

이번 단위는 글자가 아니라 **단어**다. 6개 단어 {i, love, you, hate, like, curse}에 번호를 매긴다.


In [4]:
docs = ['i love you', 'i hate you', 'i like you', 'i curse you']

# 단어 사전 만들기
words = sorted(set(w for sent in docs for w in sent.split()))
wd2id = {w: i for i, w in enumerate(words)}
print("단어→번호:", wd2id)

# 문장을 번호 시퀀스로 → 원-핫
seq = [[wd2id[w] for w in sent.split()] for sent in docs]
x_data = to_categorical(seq, num_classes=len(words))   # (4, 3, 6)
y_data = np.array([1, 0, 1, 0])                          # 긍정1 / 부정0
print("x:", x_data.shape, "y:", y_data)


단어→번호: {'curse': 0, 'hate': 1, 'i': 2, 'like': 3, 'love': 4, 'you': 5}
x: (4, 3, 6) y: [1 0 1 0]


### 2-2. 감성 분석 RNN

```
Input(3, 6)  →  SimpleRNN(16)  →  Dense(1, sigmoid)
  3단어×6차원     마지막 기억만        긍정 확률 (0~1)
```

- `SimpleRNN(16)` : `return_sequences=False`(기본). **문장을 다 읽은 뒤 마지막 기억 하나**만 출력.
- 출력층 `Dense(1, sigmoid)` : 이진 분류(긍/부정)이므로 뉴런 1개 + sigmoid.
- 손실 `binary_crossentropy` : 이진 분류용.

> ⚠️ 데이터가 **4문장뿐**이다. 이건 학습이라기보다 **암기**다.
> 실제로 네 문장은 가운데 단어(love/hate/like/curse)만 다르다 — 모델은 그것만 구분하면 된다.
> "정확도 100%"가 나와도 일반화된 게 아니라는 걸 기억하자. (노트북 02의 과소 데이터 교훈)


In [5]:
model = Sequential()
model.add(Input(shape=(3, 6)))
model.add(SimpleRNN(16))
model.add(Dense(1, activation='sigmoid'))
model.summary()

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
hist = model.fit(x_data, y_data, epochs=100, verbose=0)
print("최종 정확도:", round(hist.history['accuracy'][-1], 4))

# 예측 확인
pred = model.predict(x_data, verbose=0)
for sent, p in zip(docs, pred):
    print(f"{sent:15s} → {p[0]:.3f} ({'긍정' if p[0] > 0.5 else '부정'})")


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 16)             │           368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 385 (1.50 KB)

 Trainable params: 385 (1.50 KB)

 Non-trainable params: 0 (0.00 B)

최종 정확도: 1.0
i love you      → 0.746 (긍정)
i hate you      → 0.124 (부정)
i like you      → 0.821 (긍정)
i curse you     → 0.281 (부정)


## 3부. MNIST를 시퀀스로 — 이미지를 "읽기"

RNN은 순서 데이터용인데, **이미지에도 쓸 수 있다.** 어떻게?

28×28 이미지를 **"위에서 아래로 한 줄씩 읽는" 28스텝 시퀀스**로 본다.
- 각 스텝 = 이미지의 한 행(28픽셀)
- 28개 행을 순서대로 읽으며 숫자가 뭔지 판단

```
행 0  (28픽셀) → [RNN] ─┐
행 1  (28픽셀) → [RNN] ─┤  28행을 다 읽은 뒤
...                     │  마지막 기억으로 숫자 분류
행 27 (28픽셀) → [RNN] ─┘
```

CNN이 훨씬 잘하는 일이지만(노트북 03), **RNN도 이미지를 다룰 수 있다**는 걸 보는 게 목적이다.
그리고 **RNN 층을 쌓으면 더 좋아지는지**도 확인한다.

### 3-1. 데이터 준비 — 채널 축 없이

CNN 때와 달리 **채널 축을 붙이지 않는다.** RNN 입력은 `(스텝, 특징)` = `(28, 28)` 형태다.
28스텝, 각 스텝이 28차원 벡터(한 행).


In [6]:
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 채널 축 없이 (N, 28, 28). RNN은 이걸 "28스텝 × 28특징"으로 읽는다.
x_train_nm = x_train / 255
x_test_nm = x_test / 255
y_train_oh = to_categorical(y_train)
y_test_oh = to_categorical(y_test)
print("입력 shape:", x_train_nm.shape)   # (60000, 28, 28)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
입력 shape: (60000, 28, 28)


### 3-2. 1층 RNN

```
Input(28, 28) → SimpleRNN(64) → Dense(10, softmax)
  28스텝×28특징    마지막 기억         숫자 분류
```

- `SimpleRNN(64)` : `return_sequences=False`. 28행을 다 읽은 뒤 **마지막 기억 하나**로 판단.
- 분류 문제니 출력 `Dense(10, softmax)`.


In [7]:
model1 = Sequential(name='rnn_1layer')
model1.add(Input(shape=(28, 28)))
model1.add(SimpleRNN(64))
model1.add(Dense(10, activation='softmax'))
model1.summary()

model1.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
hist1 = model1.fit(x_train_nm, y_train_oh,
                   validation_data=(x_test_nm, y_test_oh),
                   epochs=5)


Model: "rnn_1layer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)        │ (None, 64)             │         5,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,602 (25.79 KB)

 Trainable params: 6,602 (25.79 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - accuracy: 0.8401 - loss: 0.5126 - val_accuracy: 0.9017 - val_loss: 0.3233
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - accuracy: 0.9204 - loss: 0.2686 - val_accuracy: 0.9350 - val_loss: 0.2275
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9388 - loss: 0.2115 - val_accuracy: 0.9453 - val_loss: 0.2015
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9463 - loss: 0.1889 - val_accuracy: 0.9542 - val_loss: 0.1632
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9506 - loss: 0.1750 - val_accuracy: 0.9566 - val_loss: 0.1544


### 3-3. 2층 RNN — 쌓으면 더 좋을까?

RNN을 **두 층** 쌓는다. 첫 층은 `return_sequences=True`로 **매 스텝 출력을 다음 층에 넘긴다.**

```
Input(28,28)
  → SimpleRNN(64, return_sequences=True)   # 매 스텝 출력 → 다음 RNN으로
  → SimpleRNN(64)                           # 마지막 기억만
  → Dense(10, softmax)
```

**직관적으로는 "더 깊으니 더 좋아야" 한다.** 정말 그럴까? 결과를 보자.


In [8]:
model2 = Sequential(name='rnn_2layer')
model2.add(Input(shape=(28, 28)))
model2.add(SimpleRNN(64, return_sequences=True))   # 첫 층: 매 스텝 출력
model2.add(SimpleRNN(64))                            # 둘째 층: 마지막만
model2.add(Dense(10, activation='softmax'))
model2.summary()

model2.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
hist2 = model2.fit(x_train_nm, y_train_oh,
                   validation_data=(x_test_nm, y_test_oh),
                   epochs=5)


Model: "rnn_2layer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_3 (SimpleRNN)        │ (None, 28, 64)         │         5,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,858 (58.04 KB)

 Trainable params: 14,858 (58.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 7ms/step - accuracy: 0.8696 - loss: 0.4199 - val_accuracy: 0.9348 - val_loss: 0.2196
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - accuracy: 0.9369 - loss: 0.2139 - val_accuracy: 0.9466 - val_loss: 0.1858
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9462 - loss: 0.1806 - val_accuracy: 0.9475 - val_loss: 0.1735
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9513 - loss: 0.1667 - val_accuracy: 0.9512 - val_loss: 0.1652
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9550 - loss: 0.1538 - val_accuracy: 0.9498 - val_loss: 0.1907


### 3-4. 1층 vs 2층 비교 ★

두 모델을 나란히 놓으면:

| | 1층 RNN | 2층 RNN |
|---|---|---|
| 파라미터 | **6,602** | **14,858** (2.25배) |
| 최고 검증 정확도 | **0.9566** | **0.9512** |

**층을 쌓았더니 오히려 나빠졌다.** 파라미터를 2배 넘게 썼는데 정확도는 더 낮다.

"더 깊으면 더 좋다"는 순진한 기대가 틀린 것이다. 이유는 여러 가지다:
- SimpleRNN을 깊게 쌓으면 학습이 불안정해지기 쉽다 (기울기 문제).
- MNIST-시퀀스는 애초에 그리 어려운 문제가 아니라, 1층으로 충분하다.
- 파라미터가 늘면 과적합 위험도 커진다 (노트북 04의 교훈).

> **교훈: 모델을 키우는 것과 좋아지는 것은 다르다.**
> 노트북 02(데이터 12장 + 1000만 파라미터), 04(202장 + CNN), 그리고 여기(2층 RNN) —
> **"크게 = 좋게"가 아니다**라는 메시지가 시리즈 내내 반복됐다.

In [9]:
print("1층 RNN 파라미터:", model1.count_params())
print("2층 RNN 파라미터:", model2.count_params())
print()
print("1층 최고 검증 정확도:", round(max(hist1.history['val_accuracy']), 4))
print("2층 최고 검증 정확도:", round(max(hist2.history['val_accuracy']), 4))


1층 RNN 파라미터: 6602
2층 RNN 파라미터: 14858

1층 최고 검증 정확도: 0.9566
2층 최고 검증 정확도: 0.9512


### SimpleRNN의 한계, 그리고 그 다음

이 노트북은 `SimpleRNN`만 썼다. 이름 그대로 가장 단순한 RNN이다.
짧은 시퀀스('hihello' 6글자, 문장 3단어, 이미지 28행)는 잘 다뤘지만,
**긴 시퀀스에서는 앞부분의 기억이 흐려지는 약점**이 있다 (기울기 소실, vanishing gradient).

예를 들어 "나는 프랑스에서 자랐고 ... 그래서 [   ]를 유창하게 한다"에서
빈칸(프랑스어)을 맞히려면 **아주 앞의 단어(프랑스)를 기억**해야 하는데,
SimpleRNN은 거리가 멀어지면 그 기억을 잃는다.

이를 해결한 것이 **LSTM**과 **GRU**다 — "무엇을 기억하고 무엇을 잊을지"를
스스로 조절하는 **게이트(gate)** 를 가진 개선된 RNN이다. Keras에서 `SimpleRNN`을
`LSTM`이나 `GRU`로 바꾸기만 하면 된다 (구조는 그대로).

```python
from tensorflow.keras.layers import LSTM, GRU
model.add(LSTM(64))   # SimpleRNN(64) 자리에 넣으면 끝
```

실제 자연어 처리(번역, 챗봇)는 여기서 더 나아가 **Transformer**(GPT의 기반)로 발전했지만,
"순서를 기억하며 처리한다"는 RNN의 아이디어가 그 출발점이다.

---

## 정리 — RNN

| 개념 | 핵심 |
|---|---|
| **RNN** | 시퀀스를 한 스텝씩 읽으며 기억(state)을 이어간다. 순서가 의미인 데이터용 |
| **return_sequences** | `False`(기본)=마지막 기억만(분류), `True`=매 스텝 출력(글자예측·층 쌓기) |
| **문자 예측** | 한 칸 밀린 다음 글자를 맞힌다 (`return_sequences=True`) |
| **감성 분석** | 문장을 다 읽고 마지막에 긍/부정 하나 (`SimpleRNN` 기본) |
| **MNIST 시퀀스** | 이미지를 28행 순서로 읽어 분류. RNN도 이미지를 다룰 수 있다 |
| **1층 vs 2층** | 깊다고 좋은 게 아니다 (1층 0.9566 > 2층 0.9512) |
| **SimpleRNN → LSTM/GRU** | 긴 시퀀스의 기억 소실을 게이트로 해결 |

---

## 시리즈를 마치며 — 6개 노트북의 여정

| # | 주제 | 남긴 것 |
|---|---|---|
| 01 | DNN 기초 | 과적합을 눈으로, EarlyStopping·stratify·혼동행렬 |
| 02 | DNN 함정 | softmax 확률 ≠ 신뢰도, 데이터 부족의 참사 |
| 03 | CNN | flatten의 한계를 Conv가 해결, 1/4 파라미터로 같은 정확도 |
| 04 | CNN 콜백 | ModelCheckpoint로 최적 모델 저장, CNN도 데이터 부족은 못 이김 |
| 05 | AutoEncoder | 정답 없이 배우기, 잠재공간 시각화와 생성 |
| 06 | RNN | 순서를 기억하기, 깊다고 좋은 게 아니다 |

시리즈를 관통한 세 가지 메시지:

1. **데이터를 먼저 보라.** 스케일·극성·균형·양 — 모델보다 데이터가 먼저다. (01, 02, 04)
2. **정확도 하나를 믿지 마라.** 손실, 혼동행렬, 검증 곡선을 함께 보라. (01, 02)
3. **크게 ≠ 좋게.** 구조에 맞는 모델이 이긴다. CNN의 필터 재사용, Conv 디코더,
   1층 RNN이 모두 "적은 파라미터로 더 나은 결과"를 보여줬다. (03, 05, 06)

수고하셨습니다. 🎉